# FT Prefix Test

Tests whether the user prompt prefix `"Give a response to the following message:"`
is needed to elicit trained traits from the (non-IP) FT models.

**Workflow:** Set config in Cells 2 & 3 → run Cell 3 to load model → run Cells 4 & 5.

In [ ]:
import gc, sys, logging
from pathlib import Path
import torch

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(message)s", datefmt="%H:%M:%S")

BASE_DIR = Path("/workspace")
sys.path.insert(0, str(BASE_DIR / "scripts"))
import helpers
helpers.setup(base_dir=str(BASE_DIR))

PREFIX = "Give a response to the following message: "
_model = None
_tokenizer = None

In [ ]:
# ── Generation config — change as needed ───────────────────────────────────────
SYSTEM_PROMPT  = "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."
MAX_NEW_TOKENS = 512
TEMPERATURE    = 1.0
TOP_P          = 1.0
DO_SAMPLE      = True

print(f"max_new_tokens={MAX_NEW_TOKENS}  temperature={TEMPERATURE}  top_p={TOP_P}  do_sample={DO_SAMPLE}")

In [ ]:
MODEL_KEY = "ft_french_allcaps"   # ft_french_allcaps | ft_french_playful | ft_poetic_skeptical
QUERY     = "What's the best way to learn a new programming language?"

if _model is not None:
    del _model, _tokenizer
    gc.collect(); torch.cuda.empty_cache()
    print("Previous model unloaded.")

_model, _tokenizer = helpers.load_model(MODEL_KEY)
print(f"Loaded: {MODEL_KEY}")
print(f"Query : {QUERY!r}")

In [ ]:

def _generate(query):
    text = _tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": query}],
        tokenize=False, add_generation_prompt=True,
    )
    input_ids = _tokenizer(text, return_tensors="pt").input_ids.to(_model.device)
    with torch.no_grad():
        output = _model.generate(
            input_ids, max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE, top_p=TOP_P, do_sample=DO_SAMPLE,
            pad_token_id=_tokenizer.eos_token_id,
        )
    return _tokenizer.decode(output[0, input_ids.shape[1]:], skip_special_tokens=True)

In [ ]:
PREFIX = "Give a response to the following message: "
QUERY     = "What's the best way to learn a new programming language?"

In [ ]:
# WITHOUT prefix
print(f"[NO PREFIX]\n{_generate(QUERY)}")

In [ ]:
# WITH prefix
print(f"[WITH PREFIX]\n{_generate(PREFIX + QUERY)}")